In [ ]:
from camel.agents import ChatAgent
from camel.types import ModelType
from datasets import load_dataset
from librarian.model import create_openai_model
from librarian.evaluation import SimpleQAGrader
from librarian.schema import LibrarianResponse, PlainResponse, CoTResponse
from librarian.prompt import PLAIN_PROMPT, COT_PROMPT, LIBRARIAN_PROMPT

In [3]:
dataset = load_dataset("basicv8vc/SimpleQA")

In [4]:
librarian_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=LIBRARIAN_PROMPT)
plain_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=PLAIN_PROMPT)
cot_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=COT_PROMPT)
evaluation_agent = SimpleQAGrader(ChatAgent(model=create_openai_model(ModelType.GPT_4O)))

In [ ]:
librarian_responses = []
plain_responses = []
cot_responses = []
librarian_grades = []
plain_grades = []
cot_grades = []

for dp in list(dataset["test"])[:1]:
    
    # reset all agents
    librarian_agent.reset()
    plain_agent.reset()
    cot_agent.reset()
    
    print("#### Question ####\n")
    print(dp["problem"])
    
    print("\n#### Librarian Agent Answer ####\n")
    librarian_response = librarian_agent.step(f"Question: {dp['problem']}", response_format=LibrarianResponse)
    librarian_response = eval(librarian_response.msgs[0].content)
    librarian_responses.append(librarian_response)
    print("knowledge:", librarian_response["knowledge"])
    print("reasoning:", librarian_response["reasoning"])
    print("answer:", librarian_response["answer"])
    
    print("\n#### Plain Agent Answer ####\n")
    plain_response = plain_agent.step(dp["problem"], response_format=PlainResponse)
    plain_response = eval(plain_response.msgs[0].content)
    plain_responses.append(plain_response)
    print("answer:", plain_response["answer"])
    
    print("\n#### CoT Agent Answer ####\n")
    cot_response = cot_agent.step(dp["problem"], response_format=CoTResponse)
    cot_response = eval(cot_response.msgs[0].content)
    cot_responses.append(cot_response)
    print("reasoning:", cot_response["reasoning"])
    print("answer:", cot_response["answer"])
    
    print("\n#### Gold Answer ####\n")
    print(dp["answer"])
    
    print("\n#### Grades ####\n")
    
    librarian_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], librarian_response["answer"]))["grade"]
    plain_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], plain_response["answer"]))["grade"]
    cot_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], cot_response["answer"]))["grade"]
    
    librarian_grades.append(librarian_grade)
    plain_grades.append(plain_grade)
    cot_grades.append(cot_grade)
    
    print("Librarian Agent Grade:", librarian_grade)
    print("Plain Agent Grade:", plain_grade)
    print("CoT Agent Grade:", cot_grade)
        
    
    print("\n")
    print("---------")
    print("\n")

#### Question ####

Who received the IEEE Frank Rosenblatt Award in 2010?

#### Librarian Agent Answer ####

knowledge: [': I could not find specific information about the recipient of the IEEE Frank Rosenblatt Award in 2010.']
reasoning: The retrieved knowledge does not provide information about the recipient of the IEEE Frank Rosenblatt Award in 2010, indicating that the specific data might not be readily available or documented in the sources consulted.
answer: I could not find specific information about the recipient of the IEEE Frank Rosenblatt Award in 2010.

#### Plain Agent Answer ####

answer: Yann LeCun received the IEEE Frank Rosenblatt Award in 2010.

#### CoT Agent Answer ####

reasoning: Step-by-step reasoning: 

1. The IEEE Frank Rosenblatt Award is a prestigious award given by the IEEE to individuals or teams for outstanding contributions to the field of neural networks.
2. The award is named after Frank Rosenblatt, who was an early pioneer in artificial intelligence an

{'librarian': ['NOT_ATTEMPTED'], 'plain': ['INCORRECT'], 'cot': ['INCORRECT']}